# Computational Theory: SHA-256

This notebook works through the problems set out in the assessment brief, building up an implementation of SHA-256 as described in the [Secure Hash Standard (FIPS 180-4)](https://doi.org/10.6028/NIST.FIPS.180-4).

## Problem 0: GitHub Issues

Progress on each problem below is tracked using GitHub Issues on this repository. See the [Issues tab](../../issues) for details.

## Problem 1: Representing SHA
Problem one introduces on how how SHA can be represented through various data structures within python


### 32-bit words 
FIPS 180-4 defines a word as a group of 32 or 64 bits depending on the SHA algorithm that is used ([FIPS 180-4, Section 2.1](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=9)).

32 bit words can be represented via two distinct data structures in python.
1. Plain int: Which has arbitrary precision, meaning it can grow as large as needed but never overflows. It will happily grow into a 33 bit number. To prevent this, you must force it to behave like a 32-bit word by explicitly discarding any bits above the 32nd (& 0xFFFFFFFF) after every single addition.([Python docs: Numeric Types](https://docs.python.org/3/builtins/stdtypes.html#numeric-types-int-float-complex))
2. Numpy: Provides uint32. Which is a native, fixed-size 32-bit unsigned integer. It features automatic hardware-level overflow. If a value exceeds 0xFFFFFFFF, it natively wraps around to 0 without requiring a manual mask.([Numpy docs: uint32](https://numpy.org/devdocs/reference/arrays.scalars.html#numpy.uint32))

In conclusion, 32 bit words would be best represented usings numpy's 32 bit data structure as SHA-256 relies heavily on modulo $2^{32}$ addition. When a value exceeds 0xFFFFFFFF, the extra bits must disappear, and the number must wrap back around to zero.

In [1]:
# import numpy under its conventionbal alias np
import numpy as np 
import sys

In [2]:
# Plain Python int: unlimited precision, so it grows past 32 bits.
x = 0xFFFFFFFF
x_plus_one = x + 1

print(f"int:    {x:#010x} + 1 = {x_plus_one:#x}")

# Masking with & 0xFFFFFFFF keeps only the lowest 32 bits.
print(f"masked: {x_plus_one & 0xFFFFFFFF:#010x}")

int:    0xffffffff + 1 = 0x100000000
masked: 0x00000000


As we can see the result above has 9 hex digits, meaning it needs 33 bits, one more than a word can hold. SHA 256 requires addition modulo $2^{32}$ ([FIPS 180-4, Section 6.2.2](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=27)). So with a plain `int` masking would have to be performed after each addition. The masked result, `0x00000000`, is what a true 32-bit word should give, so it's the value we expect `uint32` to produce.

In [3]:
# uint fixed size 32 bit
max_word = np.uint32(x)
# trying to overflow the 32 bit uint
y = max_word + np.uint32(1)

# print shows that its wrapped back to 0 
print(f"uint32: {max_word:#010x} + 1 = {y:#010x}")

uint32: 0xffffffff + 1 = 0x00000000


C:\Users\jhann\AppData\Local\Temp\ipykernel_21624\513019921.py:4: RuntimeWarning: overflow encountered in scalar add
  y = max_word + np.uint32(1)


The `uint32` addition seen above returns `0x00000000`. The same as the masked value from the plain `int`. No mask needed, the wrapping was done automatically. `uint32` is fixed at 32 bits, so arithmetic wraps modulo $2^{32}$, which is exactly what FIPS 180-4 requires ([Section 6.2.2](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=27)). The `RuntimeWarning` is encountered becuase NumPy assumes the overflow is a mistake. In SHA-256 this wrap around is intended, so the warning here can be ignored ([NumPy docs: overflow errors](https://numpy.org/doc/stable/user/basics.types.html#overflow-errors)). This is why SHA-256's 32-bit words will be represented as `numpy.uint32`.

### Sequence of 32-bit words 
Having chosen `numpy.uint32` to represent a single word in the subsection above, we now decide which data structure is best suited to holding a sequence of 32-bit words. Sequences of 32-bit words appear throughout SHA-256:

| Sequence | Length | FIPS 180-4 |
|---|---|---|
| Message block | 16 words | [Section 5.2.1](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=19) |
| Message schedule | 64 words |  [Section 6.2](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=26) |
| Round constants | 64 words |  [Section 4.2.2](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=16) |
| Hash value | 8 words |  [Section 5.3.3](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=23) |

Again we have two options for storing sequences of 32 bit words in python:
1. Python list: Lists are mutable sequences, meaning that they can be altered post creation using methods such as append and pop. This does not meet the requirments needed for SHA-256 as SHA-256's sequences are fixed at 16, 64 or 8 words. It also doesn't enforce what goes in this is at the discretion of the developer. This is a drawback as it means a value larger than 32 bits can be stored, which again doesnt meet the requirments of SHA-256 ([Python list docs](https://docs.python.org/3/builtins/stdtypes.html#typesseq-list)).
2. NumPy array with `dtype=np.uint32`: The NumPy array gives us fixed lentgh, which matches the SHA-256 requirment. `dtype=np.uint32` guarantees every element really is a 32-bit word ([NumPy docs: numpy.ndarray](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.html)). 

In [4]:
# demonstartes a list will accept a value too large for a 32-bit word.
words_list = [np.uint32(1), np.uint32(2)]
words_list[0] = 0x100000000 # 33 bit size value
print(words_list)

[4294967296, np.uint32(2)]


The demonstration above shows us that a list is not suitable as it accepts a value over 32 bits. Thus, meaning the value is no longer classed as a word and the modulo $2^{32}$ arithmetic SHA-256 depends on can no longer function ([Section 6.2.2](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=27)).

In [5]:
# demonstartes array with type uint32 is fixed size
words_array = np.array([1, 2], dtype=np.uint32)
try:
    words_array[0] = 0x100000000
except OverflowError as e:
    print(f"OverflowError: {e}")

OverflowError: Python int too large to convert to C long


The results above demonstrate that a NumPy array with `dtype=np.uint32` gives us our fixed element size requirement needed for SHA-256. We can see this as the `OverFlowError` was raised rather than storing a value requiring 33 bits, guaranteeing each element fits in 32 bits. This allows us to conclude that 32 bit words will be represented using NumPy arrays with `dtype=np.uint32`.

### Input messages
Input messages in FIPS 180-4 are defined by being a bit string, meaning they are an ordered sequence of 0 and 1 bits. The length of a message is denoted by ℓ, being the number of bits ([FIPS 180-4: Section 5.1.1](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=18)). Due to this, whatever format the input message is in must be converted to bits. Python's core built-in types for manipulating binary data are `bytes` and `bytearray`. `bytes` objects are immutable sequences of integers in the range of 0 <= x < 256 ([Python Docs: bytes](https://docs.python.org/3/builtins/functions.html#func-bytes)). There is no native sequence-of-bits type within Python. The smallest individual unit we can use for representation is a byte rather than bit, where each byte holds 8 bits. Python's `str` type represents sequences of Unicode characters rather than binary data([Python Docs: The String type](https://docs.python.org/3/howto/unicode.html)). Therefore, before a `str` can be used as an input message for SHA-256, it must first be encoded into a sequence of bytes using a character encoding such as UTF-8. We can therefore consider `str`, after encoding, and `bytes` as two approaches when discussing how input messages are supplied and represented in our implementation.

In [6]:
# str measures its length in characters, not bytes.
text = "seán" # non ASCII accented character
# print length of text as represented in python 
print(f"str:   {text!r}  len = {len(text)} characters")
# print encoded version of text
print(f"bytes: {text.encode('utf-8')!r}  len = {len(text.encode('utf-8'))} bytes")

# str cannot be interpreted as binary data. byte is needed
try:
    np.frombuffer(text, dtype=np.uint32)
except TypeError as e:
    print(f"TypeError: {e}")

str:   'seán'  len = 4 characters
bytes: b'se\xc3\xa1n'  len = 5 bytes
TypeError: a bytes-like object is required, not 'str'


The demonstration above outlines the pitfalls with using `str`. We first print the length of `text` in its string format and also encoded format. We can deduce that an error would be made inferring ℓ from the string as for seán it shows to be 8 * 4 = 32 bits, while the encoded version would be 8 * 5 = 40 bits, this is the actual representation of its length. This underlines that if a non-ASCII character is present in the input message its string length will differ from its encoded version. When we try pass text as a parameter into the numpy `np.frombuffer` of type `dtype=np.uint32` we get a `TypeError` highlighting a bytes-like object is required, `str` is non-applicable.

In [7]:
# bytes is a sequence of integers, each one byte, so the bit length is exact.
message = text.encode("utf-8")
print(f"bytes:    {message!r}")
print(f"elements: {list(message)}")
print(f"length:   {len(message)} bytes = {len(message) * 8} bits")
print(f"as uint8: {np.frombuffer(message, dtype=np.uint8)}")

bytes:    b'se\xc3\xa1n'
elements: [115, 101, 195, 161, 110]
length:   5 bytes = 40 bits
as uint8: [115 101 195 161 110]


The cell above encodes `text` using UTF-8 and assigns the result to `message`. We print four things: the `bytes` literal, showing the raw byte values with the non-ASCII character represented as the escape sequences \xc3\xa1; the elements as a list, confirming that a bytes object is a sequence of integers in the range 0 to 255 as documented; the length in bytes alongside the length in bits, which is the value FIPS 180-4 denotes by ℓ; and the result of np.frombuffer, which now succeeds where it failed for str. Unlike a `str`, the length of a bytes object is unambiguous. Each element is exactly one byte, so ℓ is simply len(message) * 8, with no dependence on the characters the message happens to contain. The conversion into a NumPy array also succeeds without any encoding decision being required, as the data is already binary.Note that `dtype=np.uint8` is used here rather than np.uint32, as the message is a sequence of individual bytes at this stage. Grouping bytes into 32-bit words is only possible once padding has produced blocks of a fixed size, which is addressed in a later problem. We therefore represent input messages as bytes, encoding any str input to UTF-8 before it is hashed.

### 512-bit message blocks
According to Section 5.2.1 the message along with its padding is parsed into 512-bit blocks. It also outlines that this 512-bit block may be represented using sixteen 32-bit words ([FIPS 180-4: Section 5.2.1](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=19)). 512 bits = 64 bytes = 16 words of 32 bits. We have already established 32-bit words are represented via `np.uint32` in a NumPy array. As a block is 16 such words, it follows that a block is represented via a NumPy array of 16 `uint32`. Byte structure when representing message blocks will be big-endian as per FIPS 180-4 convention; this means that the left-most byte in the sequence is the most significant, little-endian being when the right-most byte is the most significant ([FIPS 180-4: Section 3.1](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=12)). This will also be addressed in the subsequent section.

In [8]:
# stand in for padded block, gives us sequence of increasing bytes
block = bytes(range(64))

# message block is 512-bits, or 64 bytes.
print(f"block:  {len(block)} bytes = {len(block) * 8} bits")

# parse the block into 32-bit words, read big-endian as FIPS 180-4 requires.
words = np.frombuffer(block, dtype=">u4")

# the block parses to exactly 16 words
print(f"words:  {len(words)} words, dtype = {words.dtype}")

# demonstrates the words as hex, so the grouping of four bytes into each word is visible.
print(f"hex:    {[f'{w:08x}' for w in words]}")

block:  64 bytes = 512 bits
words:  16 words, dtype = >u4
hex:    ['00010203', '04050607', '08090a0b', '0c0d0e0f', '10111213', '14151617', '18191a1b', '1c1d1e1f', '20212223', '24252627', '28292a2b', '2c2d2e2f', '30313233', '34353637', '38393a3b', '3c3d3e3f']


The output confirms the block structure set out in Section 5.2.1: 64 bytes yields exactly sixteen 32-bit words. The hex values show the big-endian convention from Section 3.1 in effect, with the first byte of each group occupying the most significant position. This allows us to conclude that 512-bit blocks will be represented as a NumPy array of sixteen `uint32` values, parsed from the blocks bytes in big-endian order. 

### The final 256-bit hash value
The final 256-bit hash value is represented by eight 32-bit words([FIPS 180-4: Section 5.3.3](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=20)). $H^{(0)}$ is the initial hash value, but this value gets updated after every block, meaning the eight 32-bit words need to be represented in a data structure that is mutable ([FIPS 180-4: Section 6.2.2](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=27)). As with message blocks, this follows the representation already established for sequences of 32-bit words, giving a NumPy array of eight `uint32`. We must also cover the conversion of those eight words into 32 bytes and then into a 64-character hexadecimal string.

In [9]:
# eight initial hash values from FIPS 180-4 Section 5.3.3, written as literals.
h = np.array([
    0x6a09e667, 0xbb67ae85, 0x3c6ef372, 0xa54ff53a,
    0x510e527f, 0x9b05688c, 0x1f83d9ab, 0x5be0cd19,
], dtype=">u4")

# The hash value is eight 32-bit words, giving 256 bits in total.
print(f"words:    {len(h)} words, dtype = {h.dtype}")

# opposed to a block parsed with np.frombuffer, this array is writable.
print(f"writable: {h.flags.writeable}")

# Concatenating the eight words gives the 256-bit digest.
digest = h.tobytes()
print(f"digest:   {len(digest)} bytes = {len(digest) * 8} bits")

# final message digest is displayed as a lowercase hexadecimal string.
print(f"hex:      {digest.hex()}")
print(f"length:   {len(digest.hex())} characters")

words:    8 words, dtype = >u4
writable: True
digest:   32 bytes = 256 bits
hex:      6a09e667bb67ae853c6ef372a54ff53a510e527f9b05688c1f83d9ab5be0cd19
length:   64 characters


This output confirms to us the structure that was set out in Section 5.3.3: eight 32-bit words, giving us a total of 256 bits. The demonstration displays that the array is writable, unlike the previous block parsed with `np.frombuffer`; this is required as the value needs updating after each block. Concatenating the eight 32-bit words produces 32 bytes, that's then converted to a 64-character lowercase hexadecimal string, the form required for the message digest outlined in Problem 6 ([ianmcloughlin Github: Assessment brief, Problem 6](https://github.com/ianmcloughlin/computational-theory/blob/main/assessment.md#problem-6-complete-sha-256)). The hex output begins with `6a09e667`, matching the first word exactly, showcasing the big-endian byte order taking effect. If it were little-endian the bytes for each word would be reversed. 

The values used here are the initial hash values from Section 5.3.3 in FIPS 180-4, so the string shown is not the digest of any message. These are used to demonstrate the conversion and representation from bytes to hexadecimal. 

This allows us to conclude that the final 256-bit hash value will be represented via a NumPy array of eight `uint32` values during its computation, before finally being converted to a 64-character lowercase hexadecimal string for output. 

### Big-endian integers
As stated in Section 3.1 of FIPS 180-4 the big-endian convention is used when representing both 32 and 64-bit words, meaning the most significant bit is stored in the left-most bit position ([FIPS 180-4: Section 3.1](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=12)). Although the standard states this convention in terms of bits, bytes follows directly from it. Meaning if the most significant bit is left-most, then the most significant byte is also the left-most. This is what allows the convention to be applied in Python, where the byte is the smallest addressable unit, as previously mentioned. This rule is also not to be confused with little-endian, which is the direct reverse of big-endian; meaning the most significant bit is stored at the right-most position ([Python Docs: int to bytes](https://docs.python.org/3/builtins/stdtypes.html#int.to_bytes)). 

In the sections above we have already seen the representation of big-endian in the 512-bit message blocks and also the final hash value; we observed that byte structure is not a separate data structure but a convention of how integers are converted to bytes. Within arrays this is done through dtype `>u4`, as is already applied in the sections above. ℓ (the length of the message) is a standalone integer rather than a part of an array. For conversion of ℓ as a standalone integer it is done through the `big` argument to `int.to_bytes` and `int.from_bytes`.

In [10]:

# demos native byte order of machine
print(f'Native byte order of the machine: {sys.byteorder}')

# four bytes, or one 32-bit word
raw = bytes([0x00, 0x01, 0x02, 0x03])
print(f"bytes:        {raw.hex(' ')}")
print(f"big-endian:    {np.frombuffer(raw, dtype='>u4')[0]:08x}")
print(f"little-endian: {np.frombuffer(raw, dtype='<u4')[0]:08x}")

# one value written out in each byte order
word = 0x6a09e667
print(f"value:         {word:08x}")
print(f"big-endian:    {np.array([word], dtype='>u4').tobytes().hex(' ')}")
print(f"little-endian: {np.array([word], dtype='<u4').tobytes().hex(' ')}")

Native byte order of the machine: little
bytes:        00 01 02 03
big-endian:    00010203
little-endian: 03020100
value:         6a09e667
big-endian:    6a 09 e6 67
little-endian: 67 e6 09 6a


The demonstration above shows that the same four bytes can produce two different values depending on the byte order used, the same value produces two different byte sequences. Neither conversion is flagged as an error. Meaning an incorrect byte order could go undetected, and produce a different digest.

`sys.byteorder` reports the machine's native order as little-endian. As plain `np.uint32` follows the native order, it would give different results on different hardware. This is why the byte order is explicitily stated through the dytpe, this is why `>u4` is used for message blocks and the final hash value.

In [11]:

# length of message in bits, as established earlier
L = len(b"abc") * 8
print(f"L:             {L} bits")

# L is written into the padding as a 64-bit, or 8 byte, big-endian integer.
print(f"big-endian:    {L.to_bytes(8, 'big').hex(' ')}")
print(f"little-endian: {L.to_bytes(8, 'little').hex(' ')}")

print(f"read back:     {int.from_bytes(L.to_bytes(8, 'big'), 'big')} bits")

L:             24 bits
big-endian:    00 00 00 00 00 00 00 18
little-endian: 18 00 00 00 00 00 00 00
read back:     24 bits


This block above allows us to conclude that big-endian integers are not represented by a distint type in Python. As an integer carries no byte order of its own. Instead we see the convention applies at the conversion between integers and bytes through the dtype `>u4` for arrays of 32-bit words, as used for message blocks and the final 256-bit hash value and through the "big" argument to int.to_bytes and int.from_bytes for standalone integers such as ℓ. In both cases the byte order is stated explicitily rather than being left to the machines native order.

## Problem 2: SHA-256 Bitwise Operations

### Ch(x, y, z) function (IMPROVE THIS ANSWER LATER CITING FOUND PAPERS)
The Ch(choose) function in SHA-256 is a bitwise operation that operates on each 32-bit word. It is defined as ([FIPS 180-4: Section 4.1.2](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=15)):

$$Ch(x, y, z) = (x \wedge y) \oplus (\neg x \wedge z)$$

The operations are applied to w-bit words in [Section 3.2](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=13), and are themselves defined in [Section 2.2.2](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=10). They map directly to Python's bitwise operations which can be found here [Python docs: bitwise operators](https://docs.python.org/3/builtins/stdtypes.html#bitwise-operations-on-integer-types).
| FIPS 180-4 | Python | Operation |
|------------|--------|-----------|
| $\wedge$   | `&`     | AND       |
| $\oplus$   | `^`      | XOR       |
| $\vee$     | `\|`      | OR       |
| $SHR^n$     | `>>`      | right shift       |
| $SHL^n$     | `<<`      | left shift       |

`Ch` acts as a bitwise selector: for each of the 32 bit positions. `x` decides which of the other two words' bits to take, where `x`'s bit is 1 we take `y`'s bit and alternatively when `x`'s bit is 0 we take `z`'s bit. Hence the function name; 'choose'.

The definition produces this behaviour in two halves. The term $x \wedge y$ keeps y's bits only at the positions where `x` is 1, and zeros elsewhere. The term $\neg x \wedge z$ keeps `z`'s bits only where `x` is 0. As a given position cannot have `x` equal to both 1 and 0, the two halves are never non-zero at the same position, so the $\oplus$ simply merges them; an $\vee$ would give the same result.

Choose comes into play inside the compression function ([FIPS 180-4: Section 6.2.2](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=28)). It is called in each of the 64 rounds; this will be seen further in Problem 5. 


In [12]:
def Ch(x, y, z):
    """Choose: for each bit position, take y's bit where x is 1 and z's bit where x is 0."""
    # FIPS 180-4, equation 4.1.2: (x AND y) XOR (NOT x AND z).
    return (x & y) ^ (~x & z)

In [13]:
# Two words to choose between, shown in full 32-bit binary.
y = np.uint32(0xdeadbeef)
z = np.uint32(0x12345678)

# With every bit of x set to 1, every position selects y.
x = np.uint32(0xffffffff)
print("x all ones:")
print(f"  x  {x:032b}")
print(f"  y  {y:032b}")
print(f"  z  {z:032b}")
print(f"     {'-' * 32}")
print(f"  Ch {Ch(x, y, z):032b}  (matches y)")

# With every bit of x set to 0, every position selects z.
x = np.uint32(0x00000000)
print()
print("x all zeros:")
print(f"  x  {x:032b}")
print(f"  y  {y:032b}")
print(f"  z  {z:032b}")
print(f"     {'-' * 32}")
print(f"  Ch {Ch(x, y, z):032b}  (matches z)")

x all ones:
  x  11111111111111111111111111111111
  y  11011110101011011011111011101111
  z  00010010001101000101011001111000
     --------------------------------
  Ch 11011110101011011011111011101111  (matches y)

x all zeros:
  x  00000000000000000000000000000000
  y  11011110101011011011111011101111
  z  00010010001101000101011001111000
     --------------------------------
  Ch 00010010001101000101011001111000  (matches z)


In [14]:
y = np.uint32(0xdeadbeef)
z = np.uint32(0x12345678)

# we set all values of x to 1 
# demos where every bit of x is 1, every position chooses y.
assert Ch(np.uint32(0xffffffff), y, z) == y

# set all values of x is 0 
# demos where every bit of x is 0, every position chooses z.
assert Ch(np.uint32(0x00000000), y, z) == z

# demos a mixed case worked out by hand from the definition.
assert Ch(np.uint32(0b1100), np.uint32(0b1010), np.uint32(0b0110)) == np.uint32(0b1010)

print("All tests passed")

All tests passed


### Maj(x, y, z) function (IMPROVE THIS ANSWER LATER CITING FOUND PAPERS)
Maj is a logical bitwise function. Similar to Ch(x, y, z) it is also used in the compression loop and operates on three 32-bit words returning a single 32-bit value. But it achieves this in a different manner; for each individual bit position, the output bit matches whichever value (1 or 0) has a majority in the three 32-bit words. As defined by [FIPS 180-4: Section 4.1.2](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=15): the formula is:

$$Maj(x, y, z) = (x \wedge y) \oplus (x \wedge z) \oplus (y \wedge z)$$

The function operates as follows (consider this a single-bit position): We will assign x = 1 y = 1 and z = 0;

1. $(x \wedge y)$ : equates to 1, as the AND operator checks if both values are 1, both bits are 1, so the value is 1.
2. $(x \wedge z)$ : equates to 0, as the AND operator checks if both values are 1, both bits are different, so the value is 0.
3. $(y \wedge z)$ : equates to 0, as the AND operator checks if both values are 1, both bits are different, so the value is 0.

Following the bitwise operators in the brackets we complete the function using the $\oplus$ (XOR) operator:

$$1 \oplus 0 \oplus 0$$

 $1 \oplus 0$ = 1 -> $1 \oplus 0$ = 1

This gives us a final value of 1 for this bit position. 


In [15]:
def Maj(x, y, z):
    """Majority: for each bit position, take the majority value from the three bits."""
    # FIPS 180-4, equation 4.1.2: (x AND y) XOR (x AND z) XOR (y AND z).
    return (x & y) ^ (x & z) ^ (y & z)

In [16]:

# single bits: x = 1, y = 1, z = 0
x, y, z = np.uint32(1), np.uint32(1), np.uint32(0)
print(f"AND terms: {x & y} {x & z} {y & z}")
print(f"Maj      : {Maj(z, y, z)}")

# 32-bit words (SHA-256 initial values H0, H1, H2)
h0, h1, h2 = np.uint32(0x6A09E667), np.uint32(0xBB67AE85), np.uint32(0x3C6EF372)
print()
print(f"  h0  {h0:032b}")
print(f"  h1  {h1:032b}")
print(f"  h2  {h2:032b}")
print(f"      {'-' * 32}")
print(f"  Maj {Maj(h0, h1, h2):032b}")

AND terms: 1 0 0
Maj      : 0

  h0  01101010000010011110011001100111
  h1  10111011011001111010111010000101
  h2  00111100011011101111001101110010
      --------------------------------
  Maj 00111010011011111110011001100111


In [17]:
# assert Maj using hex values from FIPS 180-4
assert Maj(h0, h0, h2) == h0

# assert Maj using max and min values of 32-bit format
assert Maj(np.uint32(0xFFFFFFFF), np.uint32(0x00000000), np.uint32(0x00000000)) == 0x00000000

# demos a mixed case worked out by hand from the definition.
assert Maj(np.uint32(0b1100), np.uint32(0b1010), np.uint32(0b0110)) == np.uint32(0b1110)

# symmetry assertion, to show three inputs treated equally 
assert Maj(h0, h1, h2) == Maj(h2, h1, h0)

print("All tests passed")

All tests passed


#### $ROTR^n(x)$

To understand the full functionality of the sigma functions which are discussed in subsequent sections, we must also address the rotate right function. Python has no native rotate operator so a helper function will be used to achieve this. This is defined in ([FIPS 180-4: Section 3.2](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=13)):

$$ROTR^n(x) = (x \gg n) \vee (x \ll (w - n))$$

This operation shifts all bits in a 32-bit word by n number of positions to the right. Bits that are sent off the right side of the word loop back around to the beginning. This is not to be mistaken with a logical shift right which replaces any bits lost to the right hand side with zeros on the left hand side.

Uppercase Sigma contains three operations $ROTR^n(x)$: 
1. $ROTR^{2}(x)$ : The final two bits are shifted right and end up at the far left of the word, while the other thirty bits shift two places right.
2. $ROTR^{13}(x)$: The final thirteen bits are shifted right and end up at the far left of the word, while the other nineteen bits shift thirteen places right.
3. $ROTR^{22}(x)$: The final twenty-two bits are shifted right and end up at the far left of the word, while the other ten bits shift twenty-two places right.

In [18]:
def ROTR(x, n):
    """Rotates a 32-bit integer x to the right by n bits."""
    # FIPS 180-4, Section 3.2: (x shift right n) OR (x shift left (32 - n))
    return (x >> n) | (x << (32 - n))

In [19]:
# assign x a value with a bit at either end to visualise the effect of the functions
x = np.uint32(0x80000001)

# print x 32-bit size
print(f"x             : {x:032b}")

# demo the difference of rotate right and right shift 
print(f"ROTR^2        : {ROTR(x, 2):032b}")
print(f"Right shift^2 : {(x >> 2):032b}")

x             : 10000000000000000000000000000001
ROTR^2        : 01100000000000000000000000000000
Right shift^2 : 00100000000000000000000000000000


In [20]:
# assertions for rotate right
x = np.uint32(0x80000001)

# single  bit wrapping bottom to top
assert ROTR(np.uint32(0x00000001), 1) == 0x80000000

# rotating by n then 32 - n restoring the original
assert ROTR(ROTR(x, 13), 19) == x

print("All tests passed")

All tests passed


### Sigma0(x)  
Uppercase Sigma or big Sigma is used inside the compression function loop. The loop runs 64 rounds to process each block of data and iteratively updates the working values. The function is defined in [FIPS 180-4: Section 4.1.2](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=15):

$$\Sigma_0^{256}(x) = ROTR^2(x) \oplus ROTR^{13}(x) \oplus ROTR^{22}(x)$$

The function takes in a 32-bit word, performs three right rotations by 2, 13 and 22, as defined in the section above. Then it combines them using a bitwise XOR. The XOR operator here is used to achieve a higher level of cryptographic diffusion than was used in SHA-1, as SHA-1 has no equivalent $\Sigma_0$ and only applies single rotations, thus achieving faster mixing [Gilbert and Handschuh (2004), Section 3.1](https://doi.org/10.1007/978-3-540-24654-1_13). Big Sigma is one to one. This means that different inputs will always give different outputs, which matters because it means $\Sigma_0$ can never produce a collision by itself. Different words will never be squashed into the same output [Gilbert and Handschuh (2004), Section 3.3](https://doi.org/10.1007/978-3-540-24654-1_13).

$\Sigma_0$ comes into play inside the compression function ([FIPS 180-4: Section 6.2.2](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=27)). It is called in each of the 64 rounds; it is applied to working variable a and its result is added to  Maj(a, b, c), as will be seen further in Problem 5.

In [21]:
def Sigma0(x):
    """
    SHA-256 Sigma0 function XORs three right rotations of x, 
    used in the compression loop as per FIPS 180-4 4.1.2.
    """
    # FIPS 180-4, Section 4.1.2 equation 4.4: ROTR(x, 2) XOR ROTR(x, 13) XOR ROTR(x, 22).
    return ROTR(x, 2) ^ ROTR(x, 13) ^ ROTR(x, 22)

In [22]:
x = np.uint32(0x80000001)

# display rotate right by 2, 13 and 22 
print(f"ROTR^2         : {ROTR(x, 2):032b}")
print(f"ROTR^13        : {ROTR(x, 13):032b}")
print(f"ROTR^22        : {ROTR(x, 22):032b}")
print(f"                 {'-' * 32}")

# print Sigma result in binary and hex
print(f"Sigma0(32-bits): {Sigma0(x):032b}")
print(f"Sigma0(hex)    : {Sigma0(x):#010x}")

ROTR^2         : 01100000000000000000000000000000
ROTR^13        : 00000000000011000000000000000000
ROTR^22        : 00000000000000000000011000000000
                 --------------------------------
Sigma0(32-bits): 01100000000011000000011000000000
Sigma0(hex)    : 0x600c0600


In [23]:
# assertions for Sigma0

# hand-verified value from the demo above"
assert Sigma0(np.uint32(0x80000001)) == 0x600c0600

## assert Gilbert and Handschuh linearity
assert Sigma0(h0 ^ h1) == Sigma0(h0) ^ Sigma0(h1)

# zero boundary case check
assert Sigma0(np.uint32(0)) == 0

print("All tests passed")

All tests passed


### Sigma1(x)
Sigma1 also runs in each of the 64 rounds of the compression loop, the same as Sigma0, and is defined in [FIPS 180-4: Section 4.1.2](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=15):

$$\Sigma_1^{256}(x) = ROTR^6(x) \oplus ROTR^{11}(x) \oplus ROTR^{25}(x)$$

The function is almost identical to that of Sigma0, it takes in a 32-bit word, performs three right rotations, and combines them using the bitwise operation XOR. The difference lies in how far each of the three rotations shifts the bits in the word; we now rotate right by 6, 11 and 25 rather than 2, 13 and 22. For details on how the right rotation is completed please refer to the $ROTR^n(x)$ section above. Sigma1's overarching role in SHA-256 is that it is applied to the working variable `e` and the result goes into `T1` with the choose function `Ch(e, f, g)`; this will be discussed further in Problem 5.

In a study by Vaughn and Borowczak they removed each sub-function in turn from the compression function and measured the effect it had on diffusion by using the Strict Avalanche Criterion, or SAC, which requires that flipping a single input bit changes each output bit with probability one half [Vaughn and Borowczak (2024), Section 1.1](https://doi.org/10.3390/cryptography8030040). The results found that removing $\Sigma_1$ resulted in the largest negative effect on diffusion, leading to the compression function being three to four rounds behind that of the unmodified SHA-256 while removing the message schedule as a whole had little effect [Vaughn and Borowczak (2024), Section 4.3](https://doi.org/10.3390/cryptography8030040). $\Sigma_1$ and $\Sigma_0$ are identical apart from the amounts. In section five of the paper, Vaughn and Borowczak call it 'non-obvious' why their effect on diffusion differs and point to Gilbert and Handschuh's paper referenced in the $\Sigma_0$ section above, which suggested the message schedule was a significant source of diffusion. Vaughn and Borowczak measured it by removing the message schedule and found it had a minimal effect on diffusion, with the removal of $\Sigma_1$ having the largest quantifiable effect.

In [24]:
def Sigma1(x):
    """
    SHA-256 Sigma1 function XORs three right rotations of x, 
    used in the compression loop as per FIPS 180-4 4.1.2.
    """
    # FIPS 180-4, Section 4.1.2 equation 4.5: ROTR(x, 6) XOR ROTR(x, 11) XOR ROTR(x, 25).
    return ROTR(x, 6) ^ ROTR(x, 11) ^ ROTR(x, 25)

In [25]:
x = np.uint32(0x80000001)

# display rotate right by 6, 11 and 25 
print(f"ROTR^6         : {ROTR(x, 6):032b}")
print(f"ROTR^11        : {ROTR(x, 11):032b}")
print(f"ROTR^25        : {ROTR(x, 25):032b}")
print(f"                 {'-' * 32}")

# print Sigma result in binary and hex
print(f"Sigma1(32-bits): {Sigma1(x):032b}")
print(f"Sigma1(hex)    : {Sigma1(x):#010x}")

ROTR^6         : 00000110000000000000000000000000
ROTR^11        : 00000000001100000000000000000000
ROTR^25        : 00000000000000000000000011000000
                 --------------------------------
Sigma1(32-bits): 00000110001100000000000011000000
Sigma1(hex)    : 0x063000c0


In [26]:
# assertions for Sigma1

# hand-verified value
assert Sigma1(np.uint32(0x80000001)) == 0x063000c0

# assert two Sigmas produce different outputs
assert Sigma0(x) != Sigma1(x)

## assert Gilbert and Handschuh linearity
assert Sigma1(h0 ^ h1) == Sigma1(h0) ^ Sigma1(h1)

# zero boundary case check
assert Sigma1(np.uint32(0)) == 0

print("All tests passed")

All tests passed


#### $SHR^n(x)$

For the follow on segment we will discuss $\sigma_0^{256}(x)$ (sigma_0). But first we must address another helper function that will be used in its makeup. $SHR^n(x)$ represents the logical right shift operation that will be used. This is where a binary number `x` is shifted right by n positions, defined for $0 \leq n < w$, where $w = 32$ for SHA-256. The spaces that are created on the left side are filled with zeros, the bits that shift past the right-most position are discarded. We can see it defined in [FIPS 180-4: Section 3.2](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=13):

$$SHR^n(x)=x >> n$$

For example a shift right by two would go as follows (abbreviated to 8 bits for readability):
```
x          = 10000011
n          = 2
SHR^2(x)   = 00100000
```
From this example we can see the main difference between $SHR^n(x)$ and $ROTR^n(x)$. The bits pushed off the right-hand side do not wrap around; they are discarded, and the positions vacated on the left are filled with zeros. Also due to the bits being destroyed, $SHR^n(x)$ on its own is not reversible, unlike $ROTR^n(x)$.

In [27]:
def SHR(x, n):
    """Shifts an integer x by n bits"""
    # FIPS 180-4, Section 3.2: (x shift right n)
    return x >> n

In [28]:
# assign value x 
x = np.uint32(0x80000001)

# print x and also demo the hand written value from the block above
print(f"x                 : {x:032b}")
print(f"SHR^3             : {SHR(x, 3):032b}")
print(f"SHR^3(hex)        : {SHR(x, 3):#010x}")

x                 : 10000000000000000000000000000001
SHR^3             : 00010000000000000000000000000000
SHR^3(hex)        : 0x10000000


In [29]:
# assertions for shift right helper function

# assert example from demo above
assert SHR(np.uint32(0x80000001), 3) ==  0x10000000

# assert shifting right, then back left does not equal x
assert (SHR(np.uint32(0x80000001), 3) << 3) != 0x80000001

# assert three ones from the nibble are discarded
assert SHR(np.uint32(0xFFFFFFFF), 3) == 0x1FFFFFFF

# a shift of zero leaves the word unchanged
assert SHR(np.uint32(0xFFFFFFFF), 0) == 0xFFFFFFFF

print("All tests passed")

All tests passed


### sigma_0
Lowercase sigma, not to be confused with uppercase sigma ($\Sigma_0$), is a bit-level function used to scramble a 32-bit word; this is used in the makeup of the message schedule. It takes a 32-bit input, duplicates it across three rows and performs a specific set of bitwise functions on them. Specifically, a Right Rotation by 7 bits $ROTR^7(x)$, a Right Rotation by 18 bits $ROTR^{18}(x)$, and a Logical Right Shift by 3 bits $SHR^3(x)$; then finally merging the columns by performing an XOR operation. It is defined in [FIPS 180-4: Section 4.1.2](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=15):

$$\sigma_0^{256}(x) = ROTR^7(x) \oplus ROTR^{18}(x) \oplus SHR^3(x)$$

The main structural difference between $\sigma_0$ and $\Sigma_0$ and $\Sigma_1$ is that the third right rotation is replaced with a shift right, as mentioned in the section above, this means any bits that are shifted past the furthest right bit are discarded, and the left-most positions are filled with zeros. $\sigma_0$ is used in the creation of the message schedule often denoted as W, the phase that expands the initial input so it can be used in the compression rounds. This will be discussed further in Problem 4.

From the assertion in the section above we have demonstrated that SHR is a lossy operation, meaning the bits that are shifted past the right most point are destroyed. So it is perhaps surprising to people when Gilbert and Handschuh report that both $\sigma_0$ and $\sigma_1$ are one-to-one as fucntions in their own right. They came to this conclusion by treating $\sigma_0$ as a 32×32 binary matrix over GF(2) and checked its kernel, finding only the null vector (nothing except zero maps to zero), which in linear mapping is the condition for being one-to-one [Gilbert and Handschuh (2004), Section 3.3](https://doi.org/10.1007/978-3-540-24654-1_13). During the $SHR^n(x)$ assertion we saw that a shift in isolation is irreversible. The function as a whole is not, because the two rotations retain every bit of $x$ that the shift drops.

In [30]:
def sigma0(x):
    """
    SHA-256 sigma0 function XORs two right rotations of x with a right shift, 
    used in the construction of the message schedule as per FIPS 180-4 4.1.2.
    """
    # FIPS 180-4, Section 4.1.2 equation 4.6: ROTR(x, 7) XOR ROTR(x, 18) XOR SHR(x, 3).
    return ROTR(x, 7) ^ ROTR(x, 18) ^ SHR(x, 3)

In [31]:
x = np.uint32(0x80000001)

# display rotate right by 7 and 18. Shift right by 3 
print(f"ROTR^7         : {ROTR(x, 7):032b}")
print(f"ROTR^18        : {ROTR(x, 18):032b}")
print(f"SHR^3          : {SHR(x, 3):032b}")
print(f"                 {'-' * 32}")

# print sigma0 result in binary and hex
print(f"sigma0(32-bits): {sigma0(x):032b}")
print(f"sigma0(hex)    : {sigma0(x):#010x}")

ROTR^7         : 00000011000000000000000000000000
ROTR^18        : 00000000000000000110000000000000
SHR^3          : 00010000000000000000000000000000
                 --------------------------------
sigma0(32-bits): 00010011000000000110000000000000
sigma0(hex)    : 0x13006000


In [32]:
# assertions for sigma0

# hand-verified value
assert sigma0(np.uint32(0x80000001)) == 0x13006000

#assert the shift discards three ones from the first nibble
assert sigma0(np.uint32(0xFFFFFFFF)) == 0x1FFFFFFF

# assert two Sigmas produce different outputs
assert sigma0(x) != Sigma0(x)

# assert Gilbert and Handschuh linearity
assert sigma0(h0 ^ h1) == sigma0(h0) ^ sigma0(h1)

# zero boundary case check
assert sigma0(np.uint32(0)) == 0

print("All tests passed")

All tests passed


### sigma1(x)

sigma1($\sigma_1$) is the second iteration in the lowercase sigmas. Similar to the relationship between $\Sigma_0$ and $\Sigma_1$, the function is inherently the same with the major difference coming with the amount each bit is rotated or shifted. $\sigma_1$ performs a Right Rotation by 17 bits $ROTR^{17}(x)$, a Right Rotation by 19 bits $ROTR^{19}(x)$, and a Logical Right Shift by 10 bits $SHR^{10}(x)$. $\sigma_1$ is defined in [FIPS 180-4: Section 4.1.2](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=15):

$$\sigma_1^{256}(x) = ROTR^{17}(x) \oplus ROTR^{19}(x) \oplus SHR^{10}(x)$$

The most pertinent difference between the two lowercase sigma functions is where they lie in the message schedule. $\sigma_1$ mixes words from two steps back (the most recent output), while $\sigma_0$ mixes words from fifteen steps back. $\sigma_1$ discards ten bits during its shift compared to $\sigma_0$ discarding three. Both are still considered one-to-one, as seen from the argument by Gilbert and Handschuh in [Gilbert and Handschuh (2004), Section 3.3](https://doi.org/10.1007/978-3-540-24654-1_13). Despite their near-identical structure, the lowercase pair appears to contribute far less to diffusion than the uppercase pair, as discussed in the $\Sigma_1$ section above.

In [33]:
def sigma1(x):
    """
    SHA-256 sigma1 function XORs two right rotations of x with a right shift, 
    used in the construction of the message schedule as per FIPS 180-4 4.1.2.
    """
    # FIPS 180-4, Section 4.1.2 equation 4.7: ROTR(x, 17) XOR ROTR(x, 19) XOR SHR(x, 10).
    return ROTR(x, 17) ^ ROTR(x, 19) ^ SHR(x, 10)

In [34]:
x = np.uint32(0x80000001)

# display rotate right by 17 and 19. Shift right by 10 
print(f"ROTR^17        : {ROTR(x, 17):032b}")
print(f"ROTR^19        : {ROTR(x, 19):032b}")
print(f"SHR^20         : {SHR(x, 10):032b}")
print(f"                 {'-' * 32}")

# print sigma1 result in binary and hex
print(f"sigma1(32-bits): {sigma1(x):032b}")
print(f"sigma1(hex)    : {sigma1(x):#010x}")
print(f"sigma1(hex)    : {sigma1(np.uint32(0xFFFFFFFF)):#010x}")

ROTR^17        : 00000000000000001100000000000000
ROTR^19        : 00000000000000000011000000000000
SHR^20         : 00000000001000000000000000000000
                 --------------------------------
sigma1(32-bits): 00000000001000001111000000000000
sigma1(hex)    : 0x0020f000
sigma1(hex)    : 0x003fffff


In [35]:
# assertions for sigma1

# hand-verified value
assert sigma1(np.uint32(0x80000001)) ==  0x0020f000

#assert the shift discards all bits from first two nibbles and one from the third
assert sigma1(np.uint32(0xFFFFFFFF)) == 0x003fffff

# assert two Sigmas produce different outputs
assert sigma1(x) != Sigma1(x)

# assert Gilbert and Handschuh linearity
assert sigma1(h0 ^ h1) == sigma1(h0) ^ sigma1(h1)

# zero boundary case check
assert sigma1(np.uint32(0)) == 0

print("All tests passed")

All tests passed


## Problem 3: Generating the SHA-256 Constants

### Generation of Prime Numbers: primes(n)
Prime numbers are integral to two different sections of SHA-256. The eight initial hash values that were used above are the first thirty-two bits of the fractional parts of the square roots of the first eight primes, 2 to 19 ([FIPS 180-4: Section 5.3.3](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=20)). Secondly, SHA-256 uses sixty-four 32-bit constant, one in each round of the compression function, defined as $K_0$ to $K_{63}$, which will be discussed further in Problem 5 ([FIPS 180-4: Section 6.2.2](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=27)). This follows the same rules as the initial hash values but uses the first thirty-two bits of the fractional part of the cube root of one of the first sixty-four prime numbers ([FIPS 180-4: Section 4.2.2](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=16)).

The function `primes(n)` takes in a count `n` and returns the first `n` prime numbers in ascending order. The variable `n` must not be misunderstood, `n` is a count; `primes(5)` returns the first 5 prime numbers which would be `[2, 3, 5, 7, 11]`. This pattern matches what is required for SHA-256; the first 8 primes are required for the initial hash value and the first 64 for the round constants. Each candidate is verified by trial division against primes that were already identified, stopping once the prime being tested exceeds the square root of the candidate. This approach can be seen in ([GitHub: ianmcloughlin/computational-theory](https://github.com/ianmcloughlin/computational-theory/blob/main/notes/prime-numbers.ipynb)). This is sufficient as the smallest divisor of a composite number is always a prime number no larger than its square root.

Several approaches to this problem are discussed in [this thread](https://stackoverflow.com/questions/1628949/to-find-first-n-prime-numbers-in-python); a number of the answers include 1 in the output or accept odd composites, which the assertions below test against.

In [36]:
def primes(n):
    """Returns a list of the first n prime numbers in ascending order."""
    found = []          # primes found so far
    candidate = 2       # first number to test

    while len(found) < n:
        isPrime = True # boolean value that assumes prime

        # test candidate against those in array
        for c in found:
            if c * c > candidate:
                break
            if candidate % c == 0: # if c divides into it its composite
                isPrime = False
                break

        # add to array
        if isPrime:
            found.append(candidate)

        candidate += 1 #move to next candidate

    return found

In [37]:
# example case given above
print(f"primes(5)  : {primes(5)}")

# initial hash values, as seen in problem 1
print(f"primes(8)  : {primes(8)}")

# the first sixty-four, used for the round constants
print(f"primes(64) : {primes(64)}")

primes(5)  : [2, 3, 5, 7, 11]
primes(8)  : [2, 3, 5, 7, 11, 13, 17, 19]
primes(64) : [2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 53, 59, 61, 67, 71, 73, 79, 83, 89, 97, 101, 103, 107, 109, 113, 127, 131, 137, 139, 149, 151, 157, 163, 167, 173, 179, 181, 191, 193, 197, 199, 211, 223, 227, 229, 233, 239, 241, 251, 257, 263, 269, 271, 277, 281, 283, 293, 307, 311]


In [38]:
# assertions for primes(n)

# boundary case
assert primes(0) == []

# hand verified values above
assert primes(5) == [2, 3, 5, 7, 11]

# declare a count
first_64 = primes(64)
# test output length is correct
assert len(first_64) == 64

# assert function achieves the 8th prime (initial hash values)
assert primes(8)[-1] == 19
# assert function achieves the 64th prime (round constants)
assert first_64[-1] == 311

# assertion of a sorted list of the first 64 primes
assert first_64 == sorted(first_64)

print("All primes(n) assertions passed.")

All primes(n) assertions passed.


### Calculation of the cube roots of the first 64 primes

The next stage of generating the round constants is to calculate the cube roots of the first 64 primes. After this calculation is complete only the fractional part is kept. This is multiplied by $2^{32}$ and is represented in hex, which will be seen in subsequent steps ([FIPS 180-4: Section 4.2.2](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=16)).

We use the `primes(n)` function demonstrated above to gather the first 64 primes. We then use numpy's dedicated cube root function (`cbrt`) ([NumPy docs: numpy.cbrt](https://numpy.org/doc/stable/reference/generated/numpy.cbrt.html)). This will be used as opposed to `number ** (1/3)` as it avoids a fractional part of a number that is infinite, such as 0.33333. This is important as the constants are constructed from the fractional part of the cube roots. The array which these values are stored uses `float64` which is IEEE double precision; which stores numbers with 53 binary digits (bits) of accuracy. `float64` is also NumPy's default float. The largest number that will need to be stored is the cube root of 311, which is $\sqrt[3]{311} \approx 6.78$, 53 bits of accuracy minus the 3 used by the whole number leaves 50 bits for the fractional part. Each constant needs 32, leaving 18 bits to spare ([Python docs: Floating-Point Arithmetic](https://docs.python.org/3/tutorial/floatingpoint.html)).

In [39]:
# cube root of each of the first 64 primes
roots = np.cbrt(first_64)

# first three and last prime
for i in range(3):
    print(f"cbrt({first_64[i]:3d}) : {roots[i]}")

print(f"cbrt({first_64[-1]}) : {roots[-1]}")

cbrt(  2) : 1.259921049894873
cbrt(  3) : 1.4422495703074083
cbrt(  5) : 1.709975946676697
cbrt(311) : 6.7751689522733125


In [40]:
p64 = primes(64)

# One root per prime.
assert len(roots) == 64

# Cubing each root gives its prime back, which shows the roots are correct.
# See: https://github.com/ianmcloughlin/computational-theory/blob/main/notes/roots.ipynb
assert np.allclose(roots ** 3, p64)

# Hand-verified first value: the cube root of 2, which becomes K[0].
assert np.isclose(roots[0], 1.259921049894873)

# check roots are increasing in order
assert np.all(np.diff(roots) > 0)

print("All cube root assertions passed.")

All cube root assertions passed.


### Extracting the Fractional Parts

The previous section resulted in us being able to find the cube roots of the first 64 prime numbers. Each of these cube roots is made up of a whole number and a fractional part that comes after the decimal place. According to FIPS 180-4 we will only be using the fractional part of the cube root and this is the reason for its extraction. FIPS states: “These words represent the first thirty-two bits of the fractional parts of the cube roots of the first sixty-four prime numbers” ([FIPS 180-4: Section 4.2.2](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=16)).

We use the fractional part of the cube root rather than the whole number because it is better suited for both practicality and design. In the first 64 cube roots the range of the whole numbers is 1-6, each fitting in 3 bits. This is an issue as it means 64 roots share only 6 distinct integer parts. Whereas the fractional part is different for all 64 cube roots. The fractional part of a `float64` cube root is also 50-52 bits in length so FIPS builds the round constant using the first 32 bits. If there were identical round constants this would make SHA-256 vulnerable for slide attacks which exploit round constants that are identical to eachother ([Wikipedia: Round (cryptography)](https://en.wikipedia.org/wiki/Round_(cryptography))). A different constant being used in each round strengthens the algorithms resitance to these attacks.

We can extract the fractional part from the cube root by using the `%` (modulo) operator. This modulo operator gives the remainder after division. If we divide each cube root by 1 the remainder after division will be the fractional part of the cube root ([StackOverflow: Want to separate the integer part and fractional part of float number in python](https://stackoverflow.com/questions/20654141/want-to-seperate-the-integer-part-and-fractional-part-of-float-number-in-python)). For example if we take `cbrt(311) : 6.7751689522733125`, `6.7751689522733125 % 1 = .7751689522733125`. This is a hand written example that will be demonstrated and asserted below.

In [41]:
# fractional part of each cube root
fractions = roots % 1

# display prime, root, whole number, fraction
for i in range(3):
    print(f"{first_64[i]:3d}  root={roots[i]:.6f}  whole={int(roots[i])}  frac={fractions[i]:.6f}")

print(f"{first_64[-1]:3d}  root={roots[-1]:.6f}  whole={int(roots[-1])}  frac={fractions[-1]:.6f}")

  2  root=1.259921  whole=1  frac=0.259921
  3  root=1.442250  whole=1  frac=0.442250
  5  root=1.709976  whole=1  frac=0.709976
311  root=6.775169  whole=6  frac=0.775169


In [42]:
# assertions for extraction of fractional parts

# assert all extracted fractions are in between 1 and 0
assert np.all((fractions >= 0) & (fractions < 1))

# assert hand verified fraction of prime 311
assert np.isclose(fractions[-1], 0.7751689522733125)

# assert all 64 are fractional parts are distinct
assert len(set(fractions)) == 64

print("All fractional part assertions passed.")

All fractional part assertions passed.


### Converting to 32-bit unsigned integers
To convert our fractional parts into usable 32-bit we multiply by $2^{32}$ and truncate to leave us a whole number that fits into 32-bits.
Our result is stored as a `np.uint32`, as is done throughout this notebook. The final result is displayed in hexidecimal as is done in FIPS ([FIPS 180-4: Section 4.2.2](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=16)).

The multiplication turns our "first thirty-two bits of the fractional part" into a whole number. We mutiply by 2 as in binary this moves each bit one place to the left across the binary point. Raising to the power of 32 means we select the first 32 bits of the fractional section. Both of these combined into $2^{32}$ allows us to select the first 32-bits of the fractional section and move them across the binary point into a whole number. All bits past the 32-bit threshold remain after the binary point.

We then discard the leftover fractional section of our new whole number using `np.floor`. Which allows us to truncate the answer. `np.floor` rounds down to the next whole number. As we are dealing with all positive numbers this works as a truncator to strip the fractional part away. `np.floor(...).astype(...)` will strip the fractional part and cast the answer to an int. We see this pattern used in ([Course notes: *prime-numbers.ipynb*, More Efficient Algorithm](https://github.com/ianmcloughlin/computational-theory/blob/main/notes/prime-numbers.ipynb)) where a float is floored and cast to an int.

In [43]:
# move the first 32-bits across binary point
scaled = fractions * 2**32

# truncate the answers
constants = np.floor(scaled).astype(np.uint32)

In [44]:
# show how the first prime is represented thourghout the process

print(f"root      = {roots[0]:.16f}")
print(f"fraction  = {fractions[0]:.16f}")
print(f"x 2^32    = {scaled[0]:.4f}")
print(f"truncated = {int(constants[0])}")
print(f"hex       = {constants[0]:08x}")

root      = 1.2599210498948730
fraction  = 0.2599210498948730
x 2^32    = 1116352408.8405
truncated = 1116352408
hex       = 428a2f98


In [45]:
# root: the cube root of 2.
assert np.isclose(roots[0], 1.2599210498948732)

# first element of fractions matches first of roots modulo 1
assert fractions[0] == roots[0] % 1

# multiplying by a power of 2 only shifts the bits
# and dividing again gives the fraction back.
assert scaled[0] / 2**32 == fractions[0]

# truncated: the part thrown away (.8405) is between 0 and 1.
assert 0 <= scaled[0] - constants[0] < 1
assert int(constants[0]) == 1116352408

# hex: matches K[0] in FIPS 180-4, Section 4.2.2.
assert f"{constants[0]:08x}" == "428a2f98"

print("All worked-example assertions passed.")

All worked-example assertions passed.


### Displaying the constants in hexadecimal
SHA-256 constants are displayed using hexadecimal. Hex provides a clean, compact format to represent the 32-bit values. Each hex digit holds exactly 4 bits, so the 32-bit constants fit neatly into an easily readable 8-digit hexadecimal string. These constants could also be represented using binary but these would be highly prone to human error and would need 32 characters to represent each constant. FIPS publishes round constants using hexadecimal. Using the same format here allows the results to be compared directly ([FIPS 180-4: Section 4.2.2](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=16)).

Each constant will be formatted using `:08x` defined in ([Python's Format Specification Mini-Language](https://docs.python.org/3/library/string.html#format-specification-mini-language)). Each part of the formatting has a specific job; `x` means hexadecimal in lowercase which matches the FIPS round constants. The `8` represents the width of the output, being 8 hex digits. The `0` pads with leading 0's instead of spaces, without 0 constants with a leading hex digit of 0 would be affected, e.g. `0fc19dc6` would become `fc19dc6`. We see this same formatting in `bytes_to_hex` in the course notes ([ianmcloughlin, computational theory: padding.ipynb](https://github.com/ianmcloughlin/computational-theory/blob/main/notes/padding.ipynb)). The formatting used is `:02X`, we see uppercase hex used with a hex width of two digits. This is used as bytes are being formatted, and a byte is 8 bits, or two hex digits.

In [46]:
# 64 constants outputted, eight per row
# See: https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=16
hex_constants = [f"{v:08x}" for v in constants]

for i in range(0, 64, 8):
    print(' '.join(hex_constants[i:i+8]))

428a2f98 71374491 b5c0fbcf e9b5dba5 3956c25b 59f111f1 923f82a4 ab1c5ed5
d807aa98 12835b01 243185be 550c7dc3 72be5d74 80deb1fe 9bdc06a7 c19bf174
e49b69c1 efbe4786 0fc19dc6 240ca1cc 2de92c6f 4a7484aa 5cb0a9dc 76f988da
983e5152 a831c66d b00327c8 bf597fc7 c6e00bf3 d5a79147 06ca6351 14292967
27b70a85 2e1b2138 4d2c6dfc 53380d13 650a7354 766a0abb 81c2c92e 92722c85
a2bfe8a1 a81a664b c24b8b70 c76c51a3 d192e819 d6990624 f40e3585 106aa070
19a4c116 1e376c08 2748774c 34b0bcb5 391c0cb3 4ed8aa4a 5b9cca4f 682e6ff3
748f82ee 78a5636f 84c87814 8cc70208 90befffa a4506ceb bef9a3f7 c67178f2


In [47]:
# assertions for the hexadecimal display

# one string per constant
assert len(hex_constants) == 64

# assert every contents is 8 hex digits
assert all(len(h) == 8 for h in hex_constants)

# assert all are displayed in lowercase
assert all(h == h.lower() for h in hex_constants)

# assert constants that begin with 0 are unaffected
assert [h for h in hex_constants if h.startswith('0')] == ['0fc19dc6', '06ca6351']

# without the padding, K[18] prints as seven digits
assert f"{constants[18]:x}" == 'fc19dc6'

# formatting changes how the value is shown, not the value itself
assert all(int(h, 16) == c for h, c in zip(hex_constants, constants))

# first and last round constants as published (FIPS 180-4, Section 4.2.2)
assert hex_constants[0] == '428a2f98'
assert hex_constants[-1] == 'c67178f2'

print("All hexadecimal display assertions passed.")

All hexadecimal display assertions passed.


### Comparing with the constants in the Secure Hash Standard

The previous sections produced the 64 round constants from the first 64 prime numbers. In this section they will be compared with the constants that are published by NIST. As said in previous sections the constants are derived from the first thirty-two bits of the fractional parts of the cube roots of the first 64 prime numbers, as defined in ([FIPS 180-4: Section 4.2.2](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=16)). If the previous calculations were correct the two sets of values should be identical. 

The 64 published constants are copied from Section 4.2.2 as hex strings, each will be compared one by one with hex_constants. The table is copied across by hand so it is the one place a typo could affect the comparision. This will be caught as a typo will cause a filure, not a false pass, as all the values will not match. If all 64 constants match exactly the derivation reproduces the standard. It will demonstrate that the constants are not arbitrary and anyone can recalculate them from the prime numbers. This is the basis behind the idea of nothing-up-my-sleeve numbers: “any numbers which, by their construction, are above suspicion of hidden properties” ([Wikipedia: Nothing-up-my-sleeve number](https://en.wikipedia.org/wiki/Nothing-up-my-sleeve_number)). The example in the article is SHA-2's technique of calculating the initial hash values, which was seen in Problem 1. The same idea applies here to the round constants.

In [48]:
# the 64 round constants as published in FIPS 180-4, Section 4.2.2,
# transcribed from the standard.
# See: https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=16
published = [
    '428a2f98', '71374491', 'b5c0fbcf', 'e9b5dba5', '3956c25b', '59f111f1', '923f82a4', 'ab1c5ed5',
    'd807aa98', '12835b01', '243185be', '550c7dc3', '72be5d74', '80deb1fe', '9bdc06a7', 'c19bf174',
    'e49b69c1', 'efbe4786', '0fc19dc6', '240ca1cc', '2de92c6f', '4a7484aa', '5cb0a9dc', '76f988da',
    '983e5152', 'a831c66d', 'b00327c8', 'bf597fc7', 'c6e00bf3', 'd5a79147', '06ca6351', '14292967',
    '27b70a85', '2e1b2138', '4d2c6dfc', '53380d13', '650a7354', '766a0abb', '81c2c92e', '92722c85',
    'a2bfe8a1', 'a81a664b', 'c24b8b70', 'c76c51a3', 'd192e819', 'd6990624', 'f40e3585', '106aa070',
    '19a4c116', '1e376c08', '2748774c', '34b0bcb5', '391c0cb3', '4ed8aa4a', '5b9cca4f', '682e6ff3',
    '748f82ee', '78a5636f', '84c87814', '8cc70208', '90befffa', 'a4506ceb', 'bef9a3f7', 'c67178f2',
]

In [49]:
# an incomplete transcription must fail loudly rather than quietly
print(f"generated : {len(hex_constants)}")
print(f"published : {len(published)}")

# side by side comparisions including constants with leading 0's
for i in [0, 18, 30, 63]:
    g, p = hex_constants[i], published[i]
    print(f"K[{i:2d}]  generated {g}  published {p}  {'match' if g == p else 'MISMATCH'}")

# every position where the two disagree, so a typo is easy to find
mismatches = [i for i in range(64) if hex_constants[i] != published[i]]
print(f"mismatches : {len(mismatches)}")
for i in mismatches:
    print(f"K[{i:2d}]  generated {hex_constants[i]}  published {published[i]}")

generated : 64
published : 64
K[ 0]  generated 428a2f98  published 428a2f98  match
K[18]  generated 0fc19dc6  published 0fc19dc6  match
K[30]  generated 06ca6351  published 06ca6351  match
K[63]  generated c67178f2  published c67178f2  match
mismatches : 0


In [50]:
# assertions for the comparison with the Secure Hash Standard

# the transcription is complete before anything is compared against it
assert len(published) == 64

# each transcribed value is eight lowercase hex digits
assert all(len(p) == 8 for p in published)
assert all(p == p.lower() for p in published)

# the derivation reproduces the standard exactly
assert hex_constants == published

# the standard defines 64 distinct constants
assert len(set(published)) == 64

print("All comparison assertions passed.")

All comparison assertions passed.


## Problem 4: Padding and Parsing Messages

### Appending the 1 bit

The process of padding our input begins with a single bit, which is appended to the end of the input message. FIPS says: “Append the bit ‘1’ to the end of the message, followed by k zero bits, where k is the smallest, non-negative solution to the equation ℓ + 1 + k ≡ 448 mod 512” ([FIPS 180-4: Section 5.1.1](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf#page=18)). ℓ mentioned here is the length of the message in bits. k is the number of zero bits added to the block; this and the 64-bit length will be discussed in subsequent sections. The 1 bit marks where the original message ends. That boundary is recorded in the message data itself, independently of the 64-bit length field appended at the end.

`block_parse(msg)` takes the message as a `bytes` object. With a bytes object a single bit cannot be appended, so the 1 bit is added as a whole byte, `10000000` in binary, written as `0x80` in hexadecimal. The seven zeros seen here are not discarded; they become the first seven of the k zero bits from the equation. This approach can also be seen in ([ianmcloughlin, computational theory: padding.ipynb](https://github.com/ianmcloughlin/computational-theory/blob/main/notes/padding.ipynb)) in the `blocks()` function where the following computation is done `padding = block + b"\x80"`, with the comment “We have to use bytes.” This example is done for SHA-512, not SHA-256, but this step is identical because the 1 bit is appended in both SHA algorithms.

In [51]:
### Demo for prob 4

## Problem 5: The SHA-256 Compression Function

Discussion on the issue.

Aimed start date: Week begginning on 9th November.

In [52]:
### Demo for prob 5 

## Problem 6: Complete SHA-256

Discussion on the issue.

Aimed start date: Week begginning on 23rd November.

In [53]:
### Demo for prob 6